# CureFound — train R-GCN + CompGCN on Google Colab T4

This notebook trains the two GNN baselines for the CureFound drug-repurposing
comparison study, on Colab's free T4 GPU. CPU training takes ~6-10 hours; the
T4 cuts that to ~30-60 minutes total.

**Before running:** select **Runtime → Change runtime type → T4 GPU**.

What it does:
1. Clones the CureFound repo from GitHub.
2. Installs PyKEEN (~3 minutes — pulls torch + scikit-learn + pandas).
3. Trains R-GCN and CompGCN on the full 673-node KG (one production artifact each).
4. Runs leave-one-out filtered evaluation over the 16 TREATS triples for each
   model, with bootstrap-95% CIs.
5. Zips the artifacts and offers them as a single download.

Outputs you'll download (and drop into `data/artifacts/` on your local machine):
- `rgcn.npz` + `rgcn_meta.json`
- `compgcn.npz` + `compgcn_meta.json`
- `eval_report_rgcn.json`
- `eval_report_compgcn.json`


## 1. Verify GPU is connected

In [ ]:
!nvidia-smi || echo 'No GPU — switch runtime to T4 in Runtime > Change runtime type, then re-run this cell.'

## 2. Clone CureFound

Public repo, no auth needed.

In [ ]:
!rm -rf cure-found && git clone --depth 1 https://github.com/Tempest-Zero/cure-found.git
%cd cure-found/curefound
!ls scripts/

## 3. Install PyKEEN

Takes ~3 minutes the first time.

In [ ]:
!pip install -q 'pykeen>=1.10' 'torch>=2.2'
!python -c 'import pykeen, torch; print(f"pykeen {pykeen.__version__} | torch {torch.__version__} | cuda={torch.cuda.is_available()}")'

## 4. Train R-GCN + CompGCN with leave-one-out eval

Each model: one full-data training (production artifact) + 16 LOO retrains.
On T4 expect ~15-30 min per model. Keep this tab in the foreground — Colab
free tier disconnects idle sessions after ~90 min.

In [ ]:
!python scripts/train_gnns_pykeen.py --epochs 300 --dim 64 --bootstrap 2000

## 5. Inspect the eval reports

In [ ]:
import json, pathlib
for name in ('rgcn', 'compgcn'):
    p = pathlib.Path('data/artifacts') / f'eval_report_{name}.json'
    if not p.exists():
        print(f'  {p}  (missing)')
        continue
    r = json.loads(p.read_text())
    m, ci = r['metrics'], r['bootstrap_ci_95']
    print(f'\n=== {r["model"]} (n={m["n_evaluated"]}) ===')
    print(f"  MRR     : {m['mrr']:.3f}  [{ci['mrr']['lo']:.3f}, {ci['mrr']['hi']:.3f}]")
    print(f"  Hits@1  : {m['hits_at_1']:.3f}  [{ci['hits_at_1']['lo']:.3f}, {ci['hits_at_1']['hi']:.3f}]")
    print(f"  Hits@3  : {m['hits_at_3']:.3f}  [{ci['hits_at_3']['lo']:.3f}, {ci['hits_at_3']['hi']:.3f}]")
    print(f"  Hits@10 : {m['hits_at_10']:.3f}  [{ci['hits_at_10']['lo']:.3f}, {ci['hits_at_10']['hi']:.3f}]")
    print(f"  Mean    : {m['mean_rank']:.2f}  [{ci['mean_rank']['lo']:.2f}, {ci['mean_rank']['hi']:.2f}]")

## 6. Bundle artifacts and download

Drop the resulting `gnn_artifacts.zip` into `data/artifacts/` of your local
checkout (overwrites or augments existing files). Then commit + push.

In [ ]:
from google.colab import files
import shutil, pathlib
out = pathlib.Path('/content/gnn_artifacts')
out.mkdir(exist_ok=True)
for name in ('rgcn.npz', 'rgcn_meta.json', 'compgcn.npz', 'compgcn_meta.json',
             'eval_report_rgcn.json', 'eval_report_compgcn.json'):
    src = pathlib.Path('data/artifacts') / name
    if src.exists():
        shutil.copy(src, out / name)
        print(f'  bundled {name}')
    else:
        print(f'  MISSING {name}')
shutil.make_archive('/content/gnn_artifacts', 'zip', root_dir=str(out))
files.download('/content/gnn_artifacts.zip')